# Week 4 — State & Concurrency

Covers: serialisation (`json`/`pickle`) for saving/reloading agent state, `async`/`await`
in more depth for concurrent agent/tool calls, and awareness-level threading concepts.

## 1. Serialisation — saving and reloading agent state

In [ ]:
import json
from dataclasses import dataclass, field, asdict
from typing import List

@dataclass
class AgentState:
    task: str
    history: List[str] = field(default_factory=list)
    confidence: float = 0.0

state = AgentState(task="Resolve duplicate charge", history=["classified", "drafted"], confidence=0.91)

# json — human-readable, safe, works across languages/processes. Preferred for state you
# might need to inspect, log, or hand to a non-Python system.
as_json = json.dumps(asdict(state))
print("JSON:", as_json)

reloaded_dict = json.loads(as_json)
reloaded_state = AgentState(**reloaded_dict)
print("Reloaded:", reloaded_state)

In [ ]:
import pickle

# pickle — Python-specific, can serialise arbitrary objects, but NOT safe to load from an
# untrusted source (it can execute code on load). Use only for your own trusted, internal state.
blob = pickle.dumps(state)
restored = pickle.loads(blob)
print("Restored via pickle:", restored)

## 2. `async`/`await` for concurrent agent/tool calls

In [ ]:
import asyncio, time

async def call_tool(tool_name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{tool_name} result (took {delay}s)"

async def sequential_demo():
    start = time.perf_counter()
    r1 = await call_tool("database_lookup", 0.3)
    r2 = await call_tool("web_search", 0.3)
    r3 = await call_tool("policy_check", 0.3)
    elapsed = time.perf_counter() - start
    print(f"Sequential: {elapsed:.2f}s ->", r1, "|", r2, "|", r3)

asyncio.run(sequential_demo())

In [ ]:
async def concurrent_demo():
    start = time.perf_counter()
    r1, r2, r3 = await asyncio.gather(
        call_tool("database_lookup", 0.3),
        call_tool("web_search", 0.3),
        call_tool("policy_check", 0.3),
    )
    elapsed = time.perf_counter() - start
    print(f"Concurrent: {elapsed:.2f}s ->", r1, "|", r2, "|", r3)

asyncio.run(concurrent_demo())
print()
print("Same 3 calls, concurrent version takes roughly 1/3 the wall-clock time —")
print("this is exactly why multi-agent orchestrators default to async tool execution.")

## 3. Threading/multiprocessing — awareness level only

You are not expected to implement these in this programme, but you should recognise the
distinction when you see it in a library's docs:

- **`asyncio`** (used above) — one thread, cooperative multitasking. Ideal for I/O-bound work
  (network calls, waiting on an LLM response) because the CPU is idle while waiting anyway.
- **`threading`** — multiple OS threads, still limited by Python's GIL for CPU-bound work,
  but useful for blocking I/O libraries that don't support async.
- **`multiprocessing`** — separate processes, true parallelism, used for CPU-bound work
  (e.g. embedding a large batch of documents locally rather than via an API).

Rule of thumb for this programme: agent/tool/LLM calls are I/O-bound -> `asyncio` is the
right default, and it's what MCP servers in Week 5 are built on.